# Install & Import Block


In [19]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import gymnasium as gym
import numpy as np
import matplotlib as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import pygame
import time

# Set Random Seeds

In [21]:
torch.manual_seed(42)
np.random.seed(42)

# PID Controller Classic

In [22]:
class CartPolePID:
    def __init__(self, kP: float, kI : float, kD: float):
        self.kP = kP
        self.kI = kI
        self.kD = kD
        self.integral = 0.0
        self.prev_error = 0.0

    def reset(self):
        self.integral = 0.0
        self.prev_error = 0.0

    def compute_action(self, state, kP=None, kI=None, kD=None, dt=0.02) -> int:
        P_gain = kP if kP is not None else self.kP
        I_gain = kI if kI is not None else self.kI
        D_gain = kD if kD is not None else self.kD

        cart_pos = state[0]
        p_angle = state[2]
        p_vel = state[3]
        
        error = p_angle + 0.15 * cart_pos

        P = P_gain * error

        self.integral += error * dt
        I = I_gain * self.integral

        D = D_gain * p_vel

        control_f = P + I + D

        action = 1 if control_f > 0 else 0
        return action



# Tuned PID Controller


In [23]:
env = gym.make("CartPole-v1")
pid = CartPolePID(kP=120.0, kI=0.0, kD=20.0)

state, _ = env.reset(seed=42)
pid.reset()
acceptable_steps = 0

for _ in range(500):
    action = pid.compute_action(state)
    state, reward, terminated, truncated, _ = env.step(action)
    acceptable_steps += 1

    if terminated or truncated:
        break

print(acceptable_steps)

500


# Neural Network Gain Tuner

In [24]:
class GainTuner(nn.Module):
    def __init__(self, state_dim=4):
        super(GainTuner, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )
        self.softplus = nn.Softplus()

    def forward(self, state_tensor):
        raw_gains = self.network(state_tensor)
        positive_gains = self.softplus(raw_gains)

        kP = positive_gains[0] * 150.0
        kI = positive_gains[1] * 5.0
        kD = positive_gains[2] * 30.0
        return kP, kI, kD

# REINFORCE Network

In [ ]:
class GainPolicy(nn.Module):
    def __init__(self, state_dim=4, hidden_dim=32):
        super.__init__()
        self.backbone = nn.Sequential(
            nn.linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.mean_head = nn.linear(hidden_dim, 3)
        self.log_std = nn.Parameter(torch.zeros(3))
        self.softplus = nn.Softplus()

    def forward(self, state_tensor):
        features = self.backbone(state_tensor)
        raw_mean = self.mean_head(features)
        mean = self.softplus(raw_mean)
        scale = torch.exp(self.log_std)
        dist = Normal(mean, scale)

        sampled_gains = dist.rsample()
        sampled_gains = self.softplus(sampled_gains)

        kP = sampled_gains[0] * 150.0
        kI = sampled_gains[1] * 5.0
        kD = sampled_gains[2] * 30.0

        log_prob = dist.log_prob(sampled_gains).sum()
        return (kP, kI, kD), log_prob

In [ ]:
def run_episode(env, pid, policy):
    state, _ = env.reset()
    pid.reset()

    state_t = torch.tensor(state, dtype=torch.float32)
    gains, log_prob = policy(state_t)
    kp, ki, kd = [g.item() for g in gains]
    sum_reward = 0.0;

    for step in range(500):
        action = pid.compute_action(state, kP=kp, kI=ki, kD=kd)
        state, reward, terminated, truncated, _ = env.step(action)
        sum_reward += reward

        if terminated or truncated:
            break

    return sum_reward, log_prob, (kp, ki, kd)

# Initializing Environment & Agent

In [25]:
env = gym.make("CartPole-v1", render_mode="human")
pid = CartPolePID(120, 0, 20)
tuner = GainTuner()

tuner.eval()

state, _ = env.reset(seed=42)
pid.reset()

In [ ]:
for step in range(1, 501):
    state_t = torch.FloatTensor(state)
    
    with torch.no_grad():
        kp, ki, kd = tuner(state_t)

    action = pid.compute_action(
        state, 
        kP=kp.item(), 
        kI=ki.item(), 
        kD=kd.item()
    )

    state, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.02)

    if terminated or truncated:
        print(f"Episode finished after {step} steps.")
        break

env.close()

Episode finished after 500 steps.


: 

# Self Play (A D Controls)

from gymnasium.utils.play import play

env = gym.make("CartPole-v1", render_mode="rgb_array")

# Push Left (Key 'a' or Left Arrow)
# Push Right (Key 'd' or Right Arrow)
mapping = {
    (ord("a"),): 0,
    (ord("d"),): 1,
}

# Launch
play(env, keys_to_action=mapping, fps=30)